# Exercise 1 — Doctor Summary Generator

## Objective
Build a prompt that converts raw patient consultation notes into a structured doctor's summary.

## Input
- `patient_notes`
- `patient_name`

## Expected Output
- Symptoms
- Diagnosis
- Recommendation

In [50]:
%pip install groq python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [51]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError(
        "GROQ_API_KEY not found. Check your .env file in the project root."
    )

client = Groq(api_key=api_key)

print("Groq client initialized successfully.")

Groq client initialized successfully.


In [52]:
patient_name = "Rahul Sharma"

patient_notes = """
The patient reports fever for two days, a mild headache,
and a sore throat. Temperature is 38.2 degrees Celsius.
The patient reports no known drug allergies.
"""
print("Patient:", patient_name)
print("Patient notes:", patient_notes)

Patient: Rahul Sharma
Patient notes: 
The patient reports fever for two days, a mild headache,
and a sore throat. Temperature is 38.2 degrees Celsius.
The patient reports no known drug allergies.



In [53]:
prompt_template = """
You are an assistant that organizes patient notes into a concise
clinical summary for review by a qualified healthcare professional.

Use only the information provided. Do not invent symptoms, test results,
medical history, or diagnoses. If the notes do not establish a diagnosis,
say "Not established from the provided notes." Do not prescribe medication.

Patient name: {patient_name}

Patient notes:
{patient_notes}

Return the summary using exactly these three headings:

Symptoms:
Diagnosis:
Recommendation:

Keep the summary concise. Recommendations must be general and should
not replace assessment by a qualified healthcare professional.
"""

prompt = prompt_template.format(
    patient_name=patient_name,
    patient_notes=patient_notes
)

print(prompt)


You are an assistant that organizes patient notes into a concise
clinical summary for review by a qualified healthcare professional.

Use only the information provided. Do not invent symptoms, test results,
medical history, or diagnoses. If the notes do not establish a diagnosis,
say "Not established from the provided notes." Do not prescribe medication.

Patient name: Rahul Sharma

Patient notes:

The patient reports fever for two days, a mild headache,
and a sore throat. Temperature is 38.2 degrees Celsius.
The patient reports no known drug allergies.


Return the summary using exactly these three headings:

Symptoms:
Diagnosis:
Recommendation:

Keep the summary concise. Recommendations must be general and should
not replace assessment by a qualified healthcare professional.



In [54]:
models = client.models.list()

available_models = sorted(
    model.id for model in models.data
    if "text" in model.id.lower()
    or "llama" in model.id.lower()
    or "gpt-oss" in model.id.lower()
)

for model_id in available_models:
    print(model_id)

meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b


In [55]:
model="openai/gpt-oss-120b",

In [56]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": (
                "Follow the user's instructions precisely. "
                "Do not invent medical facts."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2,
    max_tokens=500
)

summary = response.choices[0].message.content.strip()

print(summary)

**Symptoms:**  
- Fever for two days (temperature 38.2 °C)  
- Mild headache  
- Sore throat  

**Diagnosis:**  
Not established from the provided notes.  

**Recommendation:**  
Monitor symptoms, maintain adequate hydration and rest, and seek further evaluation by a qualified healthcare professional if fever persists, symptoms worsen, or new concerns arise.


In [57]:
required_headings = [
    "Symptoms:",
    "Diagnosis:",
    "Recommendation:"
]

missing_headings = [
    heading for heading in required_headings
    if heading.lower() not in summary.lower()
]

if not missing_headings:
    print("Validation passed: all required sections are present.")
else:
    print("Validation failed. Missing sections:", missing_headings)

Validation passed: all required sections are present.
